<div style="border-left:5px solid #67CBEE;padding:10px 16px;background:#F7F9FB;color:#19222D;margin:12px 0"><b>AMD Radeon Cloud · YOLO26 · Day 01</b><br><span style="color:#536273">算法 → 数据 → 训练 → 评测 → 证据。每一个结论都应由实际运行结果支撑。</span></div>

# 00 / ROCm 环境预检与课程工作目录

**建议时长：10–15 分钟**　｜　学习路径：环境 → GPU 后端 → 持久化工件

> 请按顺序运行；运行结果必须来自自己的 GPU 环境，不可把示例文字当作实测数据。

## 学习目标
- 确认当前 Jupyter Kernel 的 **PyTorch 是 ROCm/HIP 版本**，不是 CPU 或 NVIDIA CUDA 版。
- 检查 GPU 设备、Ultralytics 版本和 `/workspace` 持久化路径。
- 保存不含代理凭据的环境快照供实验复现。

**重要：** 不要在已安装 AMD 定制 PyTorch 的环境直接执行 `pip install torch` 或 `pip install -U torch`。如未安装 Ultralytics，请按照 [AMD 官方集成文档](https://docs.ultralytics.com/integrations/amd/) 在匹配的 ROCm 环境中安装。

In [1]:
from pathlib import Path
import os, sys, json
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/YOLO26_ROCm_Day1_Labs')]
package = next((p for p in candidates if (p/'course/lab_utils.py').exists()), None)
if package is None:
    raise FileNotFoundError('请先将 ZIP 完整解压，并从项目目录或 notebooks/ 目录启动 Jupyter。')
sys.path.insert(0, str(package))
from course.lab_utils import (root, ensure_dirs, load_config, save_config, write_json,
    write_csv, system_info, require_rocm, sha256)
home = ensure_dirs()
cfg = load_config()
print('Project:', package, '\nPersistent output:', home)

Project: C:\Users\Rocm\Desktop\YOLO\YOLO26_ROCm_Day1_Labs 
Persistent output: C:\Users\Rocm\Desktop\YOLO\YOLO26_ROCm_Day1_Labs\yolo_course


In [2]:
from pprint import pprint
info=system_info()
info['gpu_device_nodes']={p:Path(p).exists() for p in ('/dev/kfd','/dev/dri')}
info['proxy_configured']={n:bool(os.getenv(n)) for n in ('HTTP_PROXY','HTTPS_PROXY','http_proxy','https_proxy')}
write_json(home/'reports/environment.json',info)
pprint(info)

{'gpu_device_nodes': {'/dev/dri': False, '/dev/kfd': False},
 'gpu_name': 'AMD Radeon RX 7900 XTX',
 'gpu_visible': True,
 'platform': 'Windows-11-10.0.26300-SP0',
 'proxy_configured': {'HTTPS_PROXY': False,
                      'HTTP_PROXY': False,
                      'http_proxy': False,
                      'https_proxy': False},
 'python': '3.12.16',
 'timestamp_utc': '2026-11-02T09:09:04Z',
 'torch': '2.12.0+rocm7.14.1',
 'torch_hip': '7.14.60850',
 'ultralytics': '8.4.198'}


### 理解环境信息
- `torch.version.hip != None` 表示 ROCm PyTorch 构建。
- 在 AMD GPU 上，`torch.cuda.is_available()` 也可以返回 `True`，这是 HIP 对 PyTorch CUDA API 的兼容设计。
- 训练传 `device=0`；**不要**传 `device="rocm"`。
- 云端代理只需要继承平台设置；本课不会输出代理 URL、访问令牌或秘密。

In [3]:
STRICT_ROCM = True  # 教学 GPU 实验应保持 True；不要为了通过检查而改成 False
if STRICT_ROCM:
    device = require_rocm()
else:
    device = 'cpu'
    print('CPU-only diagnostic mode; GPU experiment conclusions are NOT valid.')
print('Device argument for Ultralytics:',device)

AMD ROCm verified: AMD Radeon RX 7900 XTX HIP: 7.14.60850
Device argument for Ultralytics: 0


In [4]:
# 可选：实际在 GPU 上计算一个小矩阵，验证张量确实被送入 GPU
import torch
if STRICT_ROCM:
    x=torch.randn((128,128),device='cuda:0')
    y=x@x
    torch.cuda.synchronize()
    assert y.device.type=='cuda'  # ROCm 也报告 cuda device type
    print('HIP kernel smoke: PASS, output shape:',tuple(y.shape))

HIP kernel smoke: PASS, output shape: (128, 128)


### 冻结预训练权重（所有 A/C/E 使用相同模型）
为了避免课程中途的自动下载或不同版本导致 A/B 实验起点不一致，首次联网时缓存 `yolo26n.pt`，后续统一从 `/workspace/yolo_course/artifacts/yolo26n.pt` 读取。缓存成功后记录 SHA256。

In [5]:
import shutil
from ultralytics import YOLO
model_cache=home/'artifacts/yolo26n.pt'
if not model_cache.is_file():
    fresh=YOLO('yolo26n.pt')
    source=Path(fresh.ckpt_path).resolve()
    assert source.is_file(),f'Model download not found at {source}'
    if source!=model_cache.resolve():shutil.copy2(source,model_cache)
assert model_cache.is_file()
print('Pinned model:',model_cache,'sha256:',sha256(model_cache))
save_config({'model':'yolo26n.pt','model_sha256':sha256(model_cache)})

Pinned model:

 C:\Users\Rocm\Desktop\YOLO\YOLO26_ROCm_Day1_Labs\yolo_course\artifacts\yolo26n.pt sha256: 9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef


{'home': 'C:\\Users\\Rocm\\Desktop\\YOLO\\YOLO26_ROCm_Day1_Labs\\yolo_course',
 'model': 'yolo26n.pt',
 'dataset_yaml': '',
 'dataset_kind': '',
 'baseline_weights': '',
 'baseline_run': '',
 'model_licence_reviewed': False,
 'model_sha256': '9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef'}

### 课后问题（写进环境报告）
1. 为什么 AMD GPU 使用 `torch.cuda`？
2. `torch.version.hip` 返回 `None` 意味着什么？
3. 为什么不能在已经配好的 ROCm 环境随意覆盖 PyTorch？

**验收文件**：`reports/environment.json`。